In [ ]:
# 1. Verificar GPU y configurar entorno
import tensorflow as tf
import time
import os

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponible: {gpus}")

if gpus:
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    
    gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_name = gpu_name[0] if gpu_name else "Unknown"
    
    if "A100" in gpu_name:
        print("🚀 A100 detectada")
        BATCH_SIZE = 64
    elif "V100" in gpu_name:
        print("⚡ V100 detectada")
        BATCH_SIZE = 48
    else:
        print("🔧 T4 detectada")
        BATCH_SIZE = 32
else:
    print("⚠️ SIN GPU - Cambia el runtime a GPU!")
    print("   Runtime → Change runtime type → T4 GPU")
    BATCH_SIZE = 16

print(f"\nBatch size: {BATCH_SIZE}")

In [ ]:
# 2. Instalar dependencias y configuración
!pip install -q kaggle gdown

import numpy as np
import json
import shutil
from pathlib import Path

# Configuración optimizada para modelo robusto
CONFIG = {
    'IMG_SIZE': 224,
    'EPOCHS_PHASE1': 20,
    'EPOCHS_PHASE2': 30,
    'LEARNING_RATE_1': 1e-3,
    'LEARNING_RATE_2': 1e-5,
    'LABEL_SMOOTHING': 0.1,
    'DROPOUT_RATE': 0.4,
    'L2_REG': 0.01,
}

# Directorio unificado
UNIFIED_DIR = Path('colombia_crops_dataset')
UNIFIED_DIR.mkdir(exist_ok=True)

print("✅ Configuración lista")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
# 3. Descargar datasets - CAFÉ (el más importante para Colombia)
import urllib.request
import zipfile

print("="*60)
print("☕ DESCARGANDO DATASET DE CAFÉ")
print("="*60)

# Dataset de café con roya (el más relevante para Colombia)
# Fuente: https://www.kaggle.com/datasets/alvarole/coffee-leaves-disease

COFFEE_DIR = UNIFIED_DIR / 'coffee'
if not COFFEE_DIR.exists():
    print("📥 Descargando dataset de café desde GitHub...")
    !git clone --depth 1 https://github.com/esgario/lara2018.git coffee_temp
    
    # El dataset BRACOL tiene las siguientes clases:
    # - Healthy, Rust (Roya), Red Spider Mite, Leaf Miner, Cercospora
    COFFEE_DIR.mkdir(parents=True, exist_ok=True)
    
    # Mapear a nombres en español
    coffee_mapping = {
        'healthy': 'Cafe___Saludable',
        'rust': 'Cafe___Roya',
        'miner': 'Cafe___Minador',
        'cercospora': 'Cafe___Cercospora',
        'phoma': 'Cafe___Phoma',
        'red_spider_mite': 'Cafe___Arana_roja'
    }
    
    # Buscar imágenes en el repo clonado
    if os.path.exists('coffee_temp/dataset'):
        for folder in os.listdir('coffee_temp/dataset'):
            src = Path('coffee_temp/dataset') / folder
            if src.is_dir():
                folder_lower = folder.lower().replace(' ', '_')
                new_name = coffee_mapping.get(folder_lower, f'Cafe___{folder}')
                dst = UNIFIED_DIR / new_name
                if src.exists():
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                    count = len(list(dst.glob('*')))
                    print(f"   ✅ {new_name}: {count} imágenes")
    
    # Limpiar
    !rm -rf coffee_temp
else:
    print("✅ Dataset de café ya existe")

In [ ]:
# 4. Descargar PlantVillage (Papa, Maíz, Tomate, Pimiento)
print("="*60)
print("🥔🌽🍅 DESCARGANDO PLANTVILLAGE")
print("="*60)

PLANTVILLAGE_URL = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"

if not os.path.exists("PlantVillage-Dataset-master"):
    print("📥 Descargando PlantVillage (~250MB)...")
    start = time.time()
    urllib.request.urlretrieve(PLANTVILLAGE_URL, "plantvillage.zip")
    print(f"   Descargado en {time.time()-start:.1f}s")
    
    print("📦 Extrayendo...")
    with zipfile.ZipFile("plantvillage.zip", 'r') as z:
        z.extractall(".")
    os.remove("plantvillage.zip")

# Cultivos colombianos de PlantVillage
COLOMBIAN_CROPS = [
    'Potato',    # Papa
    'Corn',      # Maíz  
    'Tomato',    # Tomate
    'Pepper',    # Pimiento/Ají
    'Orange',    # Naranja (cítricos)
]

# Traducciones
CROP_TRANSLATIONS = {
    'Potato': 'Papa',
    'Corn': 'Maiz',
    'Tomato': 'Tomate',
    'Pepper': 'Pimiento',
    'Orange': 'Naranja',
}

DISEASE_TRANSLATIONS = {
    'healthy': 'Saludable',
    'Early_blight': 'Tizon_temprano',
    'Late_blight': 'Tizon_tardio',
    'Common_rust_': 'Roya_comun',
    'Northern_Leaf_Blight': 'Tizon_norteno',
    'Cercospora_leaf_spot Gray_leaf_spot': 'Mancha_gris',
    'Bacterial_spot': 'Mancha_bacteriana',
    'Septoria_leaf_spot': 'Septoriosis',
    'Leaf_Mold': 'Moho_foliar',
    'Target_Spot': 'Mancha_diana',
    'Spider_mites Two-spotted_spider_mite': 'Acaros',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Virus_rizado_amarillo',
    'Tomato_mosaic_virus': 'Virus_mosaico',
    'Haunglongbing_(Citrus_greening)': 'Huanglongbing',
}

PV_DIR = Path("PlantVillage-Dataset-master/raw/color")
for folder in os.listdir(PV_DIR):
    for crop in COLOMBIAN_CROPS:
        if folder.startswith(crop):
            parts = folder.split('___')
            crop_name = CROP_TRANSLATIONS.get(parts[0], parts[0])
            disease = parts[1] if len(parts) > 1 else 'healthy'
            disease_name = DISEASE_TRANSLATIONS.get(disease, disease.replace('_', ' '))
            
            new_name = f"{crop_name}___{disease_name}"
            src = PV_DIR / folder
            dst = UNIFIED_DIR / new_name
            
            if not dst.exists() and src.exists():
                shutil.copytree(src, dst)
                count = len(list(dst.glob('*')))
                print(f"   ✅ {new_name}: {count} imgs")

print("\n✅ PlantVillage procesado")

In [ ]:
# 5. Descargar dataset de Yuca/Cassava (muy importante para Colombia)
print("="*60)
print("🥬 DESCARGANDO DATASET DE YUCA (CASSAVA)")
print("="*60)

# TensorFlow Datasets tiene Cassava integrado
import tensorflow_datasets as tfds

cassava_classes = {
    0: 'Yuca___Anublo_bacterial',
    1: 'Yuca___Rayado_marron', 
    2: 'Yuca___Acaro_verde',
    3: 'Yuca___Mosaico',
    4: 'Yuca___Saludable',
}

# Descargar dataset
print("📥 Descargando Cassava desde TensorFlow Datasets...")
cassava_ds, cassava_info = tfds.load('cassava', with_info=True, as_supervised=True)

# Guardar imágenes en estructura de carpetas
for class_id, class_name in cassava_classes.items():
    (UNIFIED_DIR / class_name).mkdir(exist_ok=True)

print("📁 Guardando imágenes de yuca...")
for split in ['train', 'validation', 'test']:
    if split in cassava_ds:
        for i, (image, label) in enumerate(cassava_ds[split]):
            class_name = cassava_classes[int(label)]
            img_path = UNIFIED_DIR / class_name / f"{split}_{i}.jpg"
            tf.io.write_file(str(img_path), tf.io.encode_jpeg(image))

# Contar
for class_name in cassava_classes.values():
    count = len(list((UNIFIED_DIR / class_name).glob('*')))
    print(f"   ✅ {class_name}: {count} imgs")

print("\n✅ Yuca/Cassava procesado")

In [ ]:
# 6. Descargar dataset de Banano/Plátano
print("="*60)
print("🍌 DESCARGANDO DATASET DE BANANO/PLÁTANO")
print("="*60)

# Dataset de Kaggle - necesita credenciales
# Alternativa: usar BananaLSD de GitHub

BANANA_DIR = UNIFIED_DIR / 'banana_temp'

print("📥 Descargando BananaLSD dataset...")
!git clone --depth 1 https://github.com/Launchpad-Lab/banana-leaf-disease-classifier.git banana_temp 2>/dev/null || echo "Repo alternativo..."

# Si el repo principal no funciona, usar Kaggle manualmente
# O descargar desde: https://www.kaggle.com/datasets/shifatearman/bananalsd

banana_mapping = {
    'healthy': 'Platano___Saludable',
    'sigatoka': 'Platano___Sigatoka_negra',
    'cordana': 'Platano___Cordana',
    'pestalotiopsis': 'Platano___Pestalotiopsis',
}

# Buscar imágenes
if os.path.exists('banana_temp/data'):
    for folder in os.listdir('banana_temp/data'):
        folder_lower = folder.lower()
        new_name = banana_mapping.get(folder_lower, f'Platano___{folder}')
        src = Path('banana_temp/data') / folder
        dst = UNIFIED_DIR / new_name
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
            count = len(list(dst.glob('*')))
            print(f"   ✅ {new_name}: {count} imgs")
else:
    print("⚠️ Dataset de banano requiere descarga manual de Kaggle")
    print("   1. Ve a: kaggle.com/datasets/sujaykapadnis/banana-disease-recognition-dataset")
    print("   2. Descarga y sube el ZIP a Colab")
    print("   3. O continúa sin banano (los otros cultivos son suficientes)")

!rm -rf banana_temp 2>/dev/null
print("\n✅ Banano procesado (o saltado)")

In [ ]:
# 7. Descargar dataset de Arroz
print("="*60)
print("🌾 DESCARGANDO DATASET DE ARROZ")
print("="*60)

# Dataset de arroz desde un repo público
print("📥 Descargando dataset de arroz...")

rice_mapping = {
    'Bacterial Blight': 'Arroz___Anublo_bacterial',
    'Blast': 'Arroz___Piricularia',
    'Brown Spot': 'Arroz___Mancha_parda',
    'Tungro': 'Arroz___Tungro',
    'Healthy': 'Arroz___Saludable',
}

# Intentar clonar un repo con datos de arroz
!git clone --depth 1 https://github.com/Nirmalsarav/Rice-disease-detection.git rice_temp 2>/dev/null || true

if os.path.exists('rice_temp'):
    # Buscar carpetas de datos
    for root, dirs, files in os.walk('rice_temp'):
        for d in dirs:
            if d in rice_mapping:
                src = Path(root) / d
                dst = UNIFIED_DIR / rice_mapping[d]
                if not dst.exists():
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                    count = len(list(dst.glob('*')))
                    print(f"   ✅ {rice_mapping[d]}: {count} imgs")
    !rm -rf rice_temp
else:
    print("⚠️ Dataset de arroz no encontrado automáticamente")
    print("   Puedes agregarlo manualmente desde Kaggle")

print("\n✅ Arroz procesado")

In [ ]:
# 8. Verificar dataset unificado y balancear clases
print("="*60)
print("📊 RESUMEN DEL DATASET COLOMBIANO")
print("="*60)

# Listar todas las clases
all_classes = sorted([d.name for d in UNIFIED_DIR.iterdir() if d.is_dir()])
class_counts = {}

print(f"\n📁 {len(all_classes)} clases encontradas:\n")

total_images = 0
crops_summary = {}

for class_name in all_classes:
    class_path = UNIFIED_DIR / class_name
    count = len([f for f in class_path.iterdir() if f.is_file()])
    class_counts[class_name] = count
    total_images += count
    
    # Agrupar por cultivo
    crop = class_name.split('___')[0]
    if crop not in crops_summary:
        crops_summary[crop] = {'classes': 0, 'images': 0}
    crops_summary[crop]['classes'] += 1
    crops_summary[crop]['images'] += count
    
    print(f"  {class_name}: {count:,} imágenes")

print(f"\n{'='*60}")
print(f"📊 RESUMEN POR CULTIVO:\n")
for crop, data in sorted(crops_summary.items()):
    print(f"  🌱 {crop}: {data['classes']} clases, {data['images']:,} imágenes")

print(f"\n{'='*60}")
print(f"📷 TOTAL: {total_images:,} imágenes en {len(all_classes)} clases")
print(f"   Promedio: {total_images // len(all_classes):,} imágenes/clase")

# Guardar número de clases
NUM_CLASSES = len(all_classes)
print(f"\n✅ NUM_CLASSES = {NUM_CLASSES}")

In [ ]:
# 9. Crear generadores de datos con augmentation agresiva
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = CONFIG['IMG_SIZE']

# Augmentation fuerte para generalización
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.6, 1.4],
    channel_shift_range=30,
    fill_mode='reflect',
    validation_split=0.15
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

print(f"🔄 Creando generadores con batch_size={BATCH_SIZE}...")

train_gen = train_datagen.flow_from_directory(
    str(UNIFIED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    str(UNIFIED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
STEPS_PER_EPOCH = train_gen.samples // BATCH_SIZE
VAL_STEPS = val_gen.samples // BATCH_SIZE

print(f"\n📊 Dataset split:")
print(f"   Training: {train_gen.samples:,} imágenes ({STEPS_PER_EPOCH} steps/epoch)")
print(f"   Validation: {val_gen.samples:,} imágenes ({VAL_STEPS} steps)")
print(f"   Clases: {NUM_CLASSES}")

In [ ]:
# 10. Crear modelo MobileNetV2 robusto
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

print("🏗️ Creando modelo MobileNetV2 para cultivos colombianos...")

# Base model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

# Cabeza de clasificación robusta
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(1024, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(CONFIG['L2_REG']))(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

print(f"\n📋 Arquitectura:")
print(f"   Parámetros totales: {model.count_params():,}")
print(f"   Capas base (congeladas): {len(base_model.layers)}")
print(f"   Clases de salida: {NUM_CLASSES}")

model.summary(show_trainable=True, expand_nested=False)

In [ ]:
# 11. FASE 1: Entrenar clasificador
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import CategoricalCrossentropy

loss_fn = CategoricalCrossentropy(label_smoothing=CONFIG['LABEL_SMOOTHING'])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_1']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_colombia_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("="*60)
print("🚀 FASE 1: Entrenando clasificador (base congelada)")
print("="*60)
print(f"   Epochs: {CONFIG['EPOCHS_PHASE1']}")
print(f"   Learning rate: {CONFIG['LEARNING_RATE_1']}")
print()

start_time = time.time()

history1 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE1'],
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1
)

phase1_time = time.time() - start_time
phase1_acc = max(history1.history['val_accuracy'])

print(f"\n✅ Fase 1 completada en {phase1_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {phase1_acc:.2%}")

In [ ]:
# 12. FASE 2: Fine-tuning
print("="*60)
print("🔧 FASE 2: Fine-tuning con descongelado gradual")
print("="*60)

model.load_weights('best_colombia_phase1.keras')

# Descongelar últimas capas
base_model.trainable = True
FREEZE_UNTIL = 100

for layer in base_model.layers[:FREEZE_UNTIL]:
    layer.trainable = False

trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"   Capas descongeladas: {len(base_model.layers) - FREEZE_UNTIL}")
print(f"   Parámetros entrenables: {trainable_params:,}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_2']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_colombia_phase2.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print(f"\n   Epochs: {CONFIG['EPOCHS_PHASE2']}")
print(f"   Learning rate: {CONFIG['LEARNING_RATE_2']}")
print()

start_time = time.time()

history2 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE2'],
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1
)

phase2_time = time.time() - start_time
final_acc = max(history2.history['val_accuracy'])
final_top3 = max(history2.history['val_top3_acc'])
total_time = phase1_time + phase2_time

print(f"\n✅ Fase 2 completada en {phase2_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {final_acc:.2%}")
print(f"   Best Val Top-3 Accuracy: {final_top3:.2%}")
print(f"\n⏱️ Tiempo total: {total_time/60:.1f} minutos")

In [ ]:
# 13. Evaluación y métricas de confianza
import matplotlib.pyplot as plt

model.load_weights('best_colombia_phase2.keras')

print("="*60)
print("📊 EVALUACIÓN FINAL - MODELO COLOMBIA")
print("="*60)

val_gen.reset()
results = model.evaluate(val_gen, verbose=0)
print(f"\n🎯 Métricas:")
print(f"   Loss: {results[0]:.4f}")
print(f"   Accuracy: {results[1]:.2%}")
print(f"   Top-3 Accuracy: {results[2]:.2%}")

# Análisis de confianza
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
confidence_scores = np.max(y_pred_probs, axis=1)

print(f"\n📈 Distribución de confianza:")
print(f"   Media: {np.mean(confidence_scores):.2%}")
print(f"   Mediana: {np.median(confidence_scores):.2%}")
print(f"   >50%: {np.mean(confidence_scores > 0.5):.1%}")
print(f"   >70%: {np.mean(confidence_scores > 0.7):.1%}")
print(f"   >90%: {np.mean(confidence_scores > 0.9):.1%}")

# Gráficas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
offset = len(history1.history['accuracy'])
axes[0].plot(history1.history['accuracy'], label='Train P1')
axes[0].plot(history1.history['val_accuracy'], label='Val P1')
axes[0].plot(range(offset, offset+len(history2.history['accuracy'])), history2.history['accuracy'], label='Train P2')
axes[0].plot(range(offset, offset+len(history2.history['val_accuracy'])), history2.history['val_accuracy'], label='Val P2')
axes[0].axvline(x=offset-0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history1.history['loss'], label='Train P1')
axes[1].plot(history1.history['val_loss'], label='Val P1')
axes[1].plot(range(offset, offset+len(history2.history['loss'])), history2.history['loss'], label='Train P2')
axes[1].plot(range(offset, offset+len(history2.history['val_loss'])), history2.history['val_loss'], label='Val P2')
axes[1].axvline(x=offset-0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confianza
axes[2].hist(confidence_scores, bins=50, edgecolor='black', alpha=0.7)
axes[2].axvline(x=np.mean(confidence_scores), color='red', linestyle='--')
axes[2].set_title('Distribución de Confianza')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('colombia_training_metrics.png', dpi=150)
plt.show()

In [ ]:
# 14. Guardar modelo y labels
print("="*60)
print("💾 GUARDANDO MODELO COLOMBIA")
print("="*60)

# Guardar Keras
model.save('colombia_crop_disease.keras')
print("✅ colombia_crop_disease.keras")

# Convertir a TFLite
print("\n🔄 Convirtiendo a TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float32]

tflite_model = converter.convert()
with open('colombia_crop_disease.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = os.path.getsize('colombia_crop_disease.tflite') / (1024*1024)
print(f"✅ colombia_crop_disease.tflite ({tflite_size:.1f} MB)")

# Labels en español
class_names = list(train_gen.class_indices.keys())
labels = []

for i, name in enumerate(class_names):
    parts = name.split('___')
    crop = parts[0]
    disease = parts[1] if len(parts) > 1 else 'Saludable'
    
    labels.append({
        "id": i,
        "name": name,
        "crop": crop,
        "display": disease.replace('_', ' ')
    })

with open('colombia_crop_labels.json', 'w', encoding='utf-8') as f:
    json.dump({"labels": labels}, f, indent=2, ensure_ascii=False)

print(f"✅ colombia_crop_labels.json ({len(labels)} clases)")

print("\n📋 Cultivos incluidos:")
for crop in sorted(set(l['crop'] for l in labels)):
    crop_diseases = [l['display'] for l in labels if l['crop'] == crop]
    print(f"   🌱 {crop}: {', '.join(crop_diseases)}")

In [ ]:
# 15. Descargar archivos
print("="*60)
print("🎉 MODELO COLOMBIA COMPLETADO")
print("="*60)

print(f"""
📊 RESULTADOS:
   • Accuracy: {final_acc:.2%}
   • Top-3 Accuracy: {final_top3:.2%}
   • Confianza media: {np.mean(confidence_scores):.2%}
   • Tiempo total: {total_time/60:.1f} minutos
   • Clases: {NUM_CLASSES}

🇨🇴 CULTIVOS COLOMBIANOS INCLUIDOS:
   ☕ Café (Roya, Cercospora, Minador)
   🥬 Yuca (Mosaico, Añublo, Ácaro)
   🥔 Papa (Tizón temprano y tardío)
   🌽 Maíz (Roya, Tizón norteño)
   🍅 Tomate (10 enfermedades)
   🍌 Plátano (Sigatoka, Cordana)*
   🌾 Arroz (Piricularia, Añublo)*
   
   * Si se descargaron correctamente

📁 ARCHIVOS:
   • colombia_crop_disease.tflite ({tflite_size:.1f} MB)
   • colombia_crop_labels.json
""")

# Descargar
from google.colab import files

print("📥 Descargando archivos...")
files.download('colombia_crop_disease.tflite')
files.download('colombia_crop_labels.json')
files.download('colombia_training_metrics.png')

print("""
═══════════════════════════════════════════════════════════
📱 INSTRUCCIONES PARA ANDROID
═══════════════════════════════════════════════════════════

1. Convertir a MindSpore en Linux:
   ./converter_lite --fmk=TFLITE \
       --modelFile=colombia_crop_disease.tflite \
       --outputFile=plant_disease_model

2. Copiar a Android:
   cp plant_disease_model.ms app/src/main/assets/
   cp colombia_crop_labels.json app/src/main/assets/plant_disease_labels.json

3. Recompilar:
   ./gradlew assembleDebug

═══════════════════════════════════════════════════════════
""")